In [ ]:
# NeuroScope — one-cell end-to-end run on Kaggle (GPU T4 x2, Internet On, input: brats20 dataset)
import os, glob, shutil, subprocess
print("inputs:", os.listdir("/kaggle/input"))
if not os.path.isdir("/kaggle/working/neuroscope/scripts"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/neetii-git/neuroscope", "/kaggle/working/neuroscope"])
    if r.returncode != 0:  # fallback: the uploaded code dataset
        hits = glob.glob("/kaggle/input/**/scripts/train.py", recursive=True)
        assert hits, "neuroscope code not found: enable Internet or attach the neuroscope-code dataset"
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), "/kaggle/working/neuroscope", dirs_exist_ok=True)
%cd /kaggle/working/neuroscope
!pip install -q nibabel pyyaml 2>/dev/null | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q tests

hits = glob.glob("/kaggle/input/**/MICCAI_BraTS2020_TrainingData", recursive=True)
assert hits, "BraTS2020 dataset not found under /kaggle/input — attach awsaf49/brats20-dataset-training-validation"
RAW = hits[0]; print("RAW =", RAW)
!ls {RAW} | head -3
!python scripts/preprocess.py --raw {RAW} --cache /kaggle/working/cache
!python scripts/train.py --cache /kaggle/working/cache --out /kaggle/working/runs/base --epochs 20
!python scripts/evaluate.py --cache /kaggle/working/cache --ckpt /kaggle/working/runs/base/best.pt --out /kaggle/working/runs/base/results

import json
split = json.load(open("/kaggle/working/cache/split.json"))
demo = os.path.join(RAW, split["test"][0])
shutil.copytree(demo, "/kaggle/working/demo_case", dirs_exist_ok=True)
!cd /kaggle/working && zip -q -r neuroscope_artifacts.zip runs demo_case && ls -lh neuroscope_artifacts.zip
print(json.dumps(json.load(open("/kaggle/working/runs/base/results/summary.json")), indent=1)[:2500])
